In [3]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [4]:
DATASET_PATH = "../data/food-101/images"

CLASSES = [
    "pizza",
    "hamburger",
    "ceviche",
    "tacos",
    "steak",
    "ramen",
    "ice_cream",
    "spaghetti_bolognese",
    "fried_rice",
    "chicken_wings"
]

In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [6]:
class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}

        for cls in classes:
            class_path = os.path.join(root_dir, cls)
            for img_name in os.listdir(class_path):
                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        from PIL import Image
        
        img = Image.open(self.image_paths[idx]).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        
        label = self.labels[idx]
        return img, label

In [7]:
dataset = FoodDataset(DATASET_PATH, CLASSES, transform)

print("Total imágenes:", len(dataset))

Total imágenes: 10000


In [8]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 8000
Test: 2000


In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [10]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x


model = SimpleCNN(len(CLASSES))

In [11]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 527.9244
Epoch 2, Loss: 460.7034
Epoch 3, Loss: 398.8590
Epoch 4, Loss: 297.8185
Epoch 5, Loss: 171.3573
Epoch 6, Loss: 74.6189
Epoch 7, Loss: 29.8970
Epoch 8, Loss: 15.3096
Epoch 9, Loss: 6.6653
Epoch 10, Loss: 6.5450


In [12]:
import torch
torch.save(model.state_dict(), "modelo_food.pth")
print("Modelo guardado correctamente")

Modelo guardado correctamente
